# SoccerMaster — example notebook

This walks through the refactored `soccermaster` package: install it, fetch the
pretrained weights, load the model, run inference, and optionally run the full
game-state pipeline.

It is written to run on **Google Colab** (a GPU runtime is recommended) and should
also work on Kaggle and on a local machine.

## What is and is not tested

Be aware of the state of this fork:

- The **packaging** is verified: the package builds, installs, ships its Hydra
  configs, and `import soccermaster` needs no third-party packages at all.
- The **pipeline dependencies** are declared but the dependency set has not been
  resolved end-to-end here (no GPU available at the time this was written).
- The **model forward pass and the 3-step pipeline have not been verified** by the
  fork author. Expect to fix small things on the first real run.

Cells below are **guarded** so nothing multi-gigabyte downloads or runs by accident.
Flip the flags when you are ready.

## Prerequisite

You need access to the weights. `soccermaster prepare` handles the Hugging Face and
Google Drive ones; the `reid` and `calibration` sets are SoccerNet-hosted and must be
placed by hand (see the last section).


In [ ]:
# Install the package straight from GitHub.
#
# `pipeline` is the extra that carries everything the game-state steps import at
# module level (torchreid, prtreid, SoccerNet, poseval, pycocotools, ...). Without it
# you can still import the package and load the model, but the pipeline will raise
# ModuleNotFoundError.
#
# CAUTION for Colab/Kaggle: those runtimes ship a CUDA-matched torch, and this
# project pins `torch>=2.4,<2.6`. A plain install can therefore DOWNGRADE torch and
# break CUDA. If that happens, use the --no-deps form below and let the host keep its
# own torch.

REPO = "git+https://github.com/SurrealisticRabbit/SoccerMaster.git@main"
EXTRAS = "pipeline"  # or "" for the lean install (import + model only)

# Keep the runtime's own torch (recommended on Colab/Kaggle):
# !pip install -q --no-deps "soccermaster[{EXTRAS}] @ {REPO}"

!pip install -q "soccermaster[{EXTRAS}] @ {REPO}"


## 1. Environment and where things get stored

Everything the package reads and writes lives under a single root —
`SOCCERMASTER_HOME`. You must set it **before** the paths are first used so that
weights and outputs land somewhere you can find (and, on Colab, somewhere you know
is ephemeral unless you mount Drive).

Run this cell, then **restart the runtime** if you set a new value, so every
subsequent import sees it.


In [ ]:
import os
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Where should weights/, data/ and outputs/ live?
# ---------------------------------------------------------------------------
USE_DRIVE = False  # set True on Colab to persist weights across sessions

try:
    import google.colab  # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if USE_DRIVE and ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    SM_HOME = Path("/content/drive/MyDrive/soccermaster")
elif ON_COLAB:
    # /content is wiped when the runtime recycles -- fine for a one-off run.
    SM_HOME = Path("/content/soccermaster")
else:
    SM_HOME = Path.cwd() / "soccermaster_home"

SM_HOME.mkdir(parents=True, exist_ok=True)
os.environ["SOCCERMASTER_HOME"] = str(SM_HOME)

print(f"python      : {sys.version.split()[0]}")
print(f"on colab    : {ON_COLAB}")
print(f"SM_HOME     : {SM_HOME}")

try:
    import torch

    print(f"torch       : {torch.__version__}")
    print(f"cuda avail  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"gpu         : {torch.cuda.get_device_name(0)}")
except ImportError:
    print("torch       : NOT INSTALLED -- install a CUDA build before continuing")


## 2. Verify the install

`import soccermaster` is deliberately dependency-free: the public API is re-exported
lazily, so this import succeeds even if nothing else is installed. That makes it a
quick check that the package itself landed.


In [ ]:
import soccermaster as sm

print("soccermaster :", sm.__version__)
print("home         :", sm.paths.home())
print("weights      :", sm.paths.weights_dir())
print("data         :", sm.paths.data_dir())
print("outputs      :", sm.paths.outputs_dir())

# The public surface: everything a notebook should need.
print("api          :", ", ".join(sm.__all__))


## 3. What weights are needed

`soccermaster prepare` reads a manifest and puts each set where the configs expect
it. Two entries cannot be fetched automatically and say so.


In [ ]:
from soccermaster.downloads import load_manifest

manifest = load_manifest()
print(f"{'name':<14}{'kind':<8}{'source':<9}destination")
print("-" * 78)
for name, entry in manifest.items():
    kind = "auto" if entry.automatic else "MANUAL"
    print(f"{name:<14}{kind:<8}{entry.source:<9}{entry.dest}")

# Which of them are already on disk?
print("\nAlready present:")
for name, entry in manifest.items():
    mark = "yes" if entry.dest.exists() and any(entry.dest.iterdir()) else "no "
    print(f"  {name:<14}{mark}")


## 4. Download the weights

Multi-gigabyte. The cell is guarded by `DOWNLOAD`; the `dry_run` pass below prints
exactly what would be fetched without touching the network.

For **inference only** you need `siglip` (backbone) + `soccermaster` (checkpoints).
Add `yolo` for step 1 of the pipeline, `qwen7b` for step 3, `sam2` for step 2.


In [ ]:
DOWNLOAD = False  # flip to True to actually fetch the weights

# What would happen, without downloading anything:
for name in ("siglip", "soccermaster", "yolo"):
    sm.prepare(name, dry_run=True)

if DOWNLOAD:
    sm.prepare("siglip")        # SigLIP2 backbone (Google) -- needed to build the model
    sm.prepare("soccermaster")  # SoccerMaster checkpoints  -- needed for inference
    sm.prepare("yolo")          # person detector          -- needed for pipeline step 1
    sm.prepare("qwen7b")        # jersey/role (Alibaba)    -- needed for pipeline step 3
    # sm.prepare("sam2")        # SAM 2 checkpoint         -- needed for pipeline step 2
else:
    print("DOWNLOAD is False - nothing fetched.")


## 5. Inspect the config (no weights needed)

Cheap sanity check. This also shows how paths are resolved: `CKPT_PATH` is rewritten
to point under `weights_dir()`, so the configs no longer depend on the working
directory.


In [ ]:
cfg = sm.load_config("pretrain")

print(f"config keys : {len(cfg)}")
for key in (
    "SIGLIP_BACKBONE_TYPE",
    "NUM_FRAMES",
    "AUG_MAX_SIZE",
    "TEMPORAL_START_LAYER",
    "CKPT_PATH",
    "TEXT_ENCODER_CKPT_PATH",
):
    print(f"{key:<22}: {cfg.get(key)}")

print("\nDATASETS_TO_HEADS (task groups -> heads):")
for dataset, heads in cfg["DATASETS_TO_HEADS"].items():
    print(f"  {dataset:<24} {heads}")


## 6. Load the model

`load_model()` builds `MultiTaskingModel` and loads the backbone plus the task heads
from `weights/SoccerMaster`. Guarded so the cell explains itself instead of raising a
confusing error when the weights are absent.


In [ ]:
model = None

checkpoint_dir = sm.paths.weights_dir() / "SoccerMaster"
backbone_dir = cfg["CKPT_PATH"]

missing = [p for p in (checkpoint_dir, backbone_dir) if not Path(p).exists()]

if missing:
    print("Weights not found yet:")
    for p in missing:
        print(f"  {p}")
    print("\nSet DOWNLOAD = True in the download cell and re-run it, or place the")
    print("files manually. The rest of the notebook needs these.")
else:
    model = sm.load_model()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"model      : {type(model).__name__}")
    print(f"parameters : {n_params/1e6:.1f} M")
    print(f"device     : {next(model.parameters()).device}")
    print(f"heads      : {list(model.multi_task_head.keys())}")


## 7. Run inference

`predict(model, frames, task=...)` takes either:

- a directory — `.../SNGS-10001` or `.../SNGS-10001/img1` (a flat folder of images
  also works), or
- an array of `uint8` HWC frames, or a list of such arrays.

It returns the **raw head outputs** (no NMS, no bbox rescaling), matching the
upstream `inference_demo.py`. `task` is one of `detection`, `lines`, `keypoints`,
`caption`, `classification` or `all`.

The cell below uses random frames so you can exercise the forward pass without any
dataset. Point `FRAMES` at a real sequence to get meaningful output.


In [ ]:
import numpy as np

# Point this at a real sequence when you have one, e.g.
#   sm.paths.data_dir() / "SoccerNetGS" / "sn500" / "SNGS-10001"
FRAMES = None

if model is None:
    print("Load the model first (previous cell).")
else:
    if FRAMES is None:
        # The temporal encoder expects cfg["NUM_FRAMES"] frames.
        n = cfg["NUM_FRAMES"]
        FRAMES = np.random.randint(0, 256, size=(n, 540, 960, 3), dtype=np.uint8)
        print(f"using {n} random frames at {FRAMES.shape[1]}x{FRAMES.shape[2]}")

    try:
        outputs = sm.predict(model, FRAMES, task="detection")
    except Exception as exc:  # surface the real error, this path is unverified
        print(f"predict failed: {type(exc).__name__}: {exc}")
    else:
        def describe(value, indent="    "):
            import torch

            if isinstance(value, torch.Tensor):
                print(f"{indent}shape={tuple(value.shape)} dtype={value.dtype}")
            elif isinstance(value, dict):
                for k, v in value.items():
                    print(f"{indent}{k}:")
                    describe(v, indent + "  ")
            elif isinstance(value, (list, tuple)):
                print(f"{indent}{type(value).__name__} of length {len(value)}")
                for i, item in enumerate(value[:2]):
                    print(f"{indent}  [{i}]")
                    describe(item, indent + "    ")
            else:
                print(f"{indent}{value}")

        for head, head_output in outputs.items():
            print(f"[{head}]")
            describe(head_output)


## 8. The full game-state pipeline (optional)

Three steps:

| step | what it does | needs |
|---|---|---|
| 1 | person detection + tracking | `yolo` weights |
| 2 | SAM 2 mask refinement | `sam2` extra + `sam2` weights + CUDA |
| 3 | calibration, jersey number, role, team | `qwen7b` weights, `reid` + `calibration` weights |

This is the same Hydra configuration the upstream scripts used, driven for you: the
dataset path, the weight directory and the state files between steps are injected, so
it no longer depends on the working directory or on hardcoded sequence ids.

Steps 1 and 3 run on CPU or GPU. **Step 2 requires the `sam2` extra and a CUDA
build.** The dataset must be in SoccerNet layout:
`<dataset>/<split>/<sequence>/img1/*.jpg` plus `sequences_info.json`.


In [ ]:
RUN_PIPELINE = False  # flip to True once the dataset and weights are in place

DATASET = sm.paths.data_dir() / "SoccerNetGS"   # <split>/<sequence>/img1 layout
OUT_DIR = sm.paths.outputs_dir() / "pipeline"
VIDEO_IDS = [10001, 10002]                      # sequences to process
STEPS = (1, 3)                                  # skip 2 unless the sam2 extra is installed

if RUN_PIPELINE:
    if not DATASET.is_dir():
        print(f"dataset not found: {DATASET}")
        print("Run: python -m soccermaster.model.data.extract_frames --help")
    else:
        # device is a CUDA_VISIBLE_DEVICES value, e.g. "0" or "0,1"
        out = sm.run_pipeline(
            DATASET,
            out_dir=OUT_DIR,
            steps=STEPS,
            video_ids=VIDEO_IDS,
            device="0",
        )
        print("outputs written to:", out)
        for path in sorted(out.rglob("*.pklz")):
            print(" ", path.relative_to(out), f"({path.stat().st_size/1e6:.1f} MB)")
else:
    print("RUN_PIPELINE is False - nothing run.")
    print(f"would read  : {DATASET}")
    print(f"would write : {OUT_DIR}")


## Troubleshooting

**`ModuleNotFoundError: No module named 'torchreid'` (or `prtreid`, `SoccerNet`, `poseval`, `pycocotools`)**
You installed without the `pipeline` extra. The base install only covers import +
model; the pipeline deps live in `.[pipeline]`.

**CUDA stopped working after the install**
The full install can downgrade torch (this project pins `torch>=2.4,<2.6`) below what
Colab/Kaggle ships. Reinstall with `--no-deps` and provide the deps yourself, or pin
the torch you want first.

**`FileNotFoundError` for `siglip2-large-patch16-512` or `SoccerMaster`**
The weights are not there. Run the download cell with `DOWNLOAD = True` (needs
`pip install huggingface_hub`), or place the files under `sm.paths.weights_dir()`.

**Step 1 complains about the YOLO checkpoint**
The configs look for `weights/yolo/yolo_v8x6_person_lr_default_best.pt`. The manifest
fetches `yolo_v8x6_finetuned.pt` from Hugging Face and renames it to that name; check
the file actually landed.

**`reid` / `calibration` are missing**
These are `manual` in the manifest and print upstream instructions rather than
downloading. They come from the
[SoccerNet GSR dev kit](https://github.com/soccernet/sn-gamestate). Put them under
`weights/reid/` and `weights/calibration/`.

**Step 2 will not start**
Step 2 needs the `sam2` extra *and* the vendored SAM 2 library:
```bash
pip install -e ".[sam2]"
pip install -e third_party/sam2     # only in a source checkout
```
and a CUDA build. On a CPU-only runtime, run `steps=(1, 3)` instead.

**Where do I look when something breaks?**
`PROVENANCE.md` maps every folder to its upstream project. If a failure is inside
`tracklab/`, `soccernet_gamestate/` or `sam2_refine/`, it is vendored third-party code
and the upstream project or its own docs in `docs/` is the better reference.

## Useful bits

```bash
soccermaster --help          # list subcommands
soccermaster prepare --list  # the weight manifest
soccermaster pipeline --dataset ... --steps 1,3 --videos 10001
soccermaster predict --frames path/to/img1 --task detection
```

`run_pipeline` forwards extra Hydra overrides, so anything in the upstream configs can
still be tweaked, e.g. `extra_overrides=["dataset.nframes=20"]`.
